# 1장 3강: 데이터 특성에 따른 가설검정 기법 선택 — 실습문제

## 실습 목표

- 비교할 변수의 척도와 집단 관계를 확인할 수 있다.
- Shapiro-Wilk 검정으로 정규성을 확인할 수 있다.
- Levene 검정으로 등분산성을 확인할 수 있다.
- 가정 점검 결과에 따라 독립표본 t검정과 Welch t검정을 선택할 수 있다.
- 선택한 검정의 p-value를 해석하여 데이터에 근거한 결론을 작성할 수 있다.

## 실습 환경 / 데이터

- Python
- pandas
- scipy.stats
- `ames_housing.csv`

| 컬럼 | 의미 | 유형 |
|---|---|---|
| `SalePrice` | 주택 판매가격 | 연속형 |
| `KitchenQual` | 주방 품질 | 범주형·순서형 |
| `HeatingQC` | 난방 품질 | 범주형·순서형 |

> 모든 판단의 유의수준은 `α = 0.05`입니다.  
> 표본 추출에는 `random_state=42`를 사용하여 실행할 때마다 같은 결과가 나오도록 합니다.


## 실습 준비

1. pandas와 `scipy.stats`를 불러오세요.
2. Ames Housing 데이터를 `df`에 불러오세요.
3. 데이터 크기, 전체 결측치 수, 컬럼명과 상위 5개 행을 확인하세요.

In [4]:
# 실습 준비 코드를 작성하세요.


# 1. 필요한 라이브러리 불러오기
import pandas as pd
from scipy import stats

# 2. Ames Housing 데이터를 df에 불러오기
df = pd.read_csv('ames_housing.csv')

print(df.isna().sum())

# 3. 데이터의 행과 열 개수, 컬럼명, 상위 5개 행 확인
print("행, 열 개수:", df.shape)
print()
print("컬럼명:", df.columns.tolist())
print()
print("상위 5개 행:")
print(df.head())

SalePrice       0
GrLivArea       0
LotArea         0
OverallQual     0
KitchenQual     0
CentralAir      0
HeatingQC       0
PavedDrive      0
Neighborhood    0
YearBuilt       0
dtype: int64
행, 열 개수: (1460, 10)

컬럼명: ['SalePrice', 'GrLivArea', 'LotArea', 'OverallQual', 'KitchenQual', 'CentralAir', 'HeatingQC', 'PavedDrive', 'Neighborhood', 'YearBuilt']

상위 5개 행:
   SalePrice  GrLivArea  LotArea  OverallQual KitchenQual CentralAir  \
0     208500       1710     8450            7          Gd          Y   
1     181500       1262     9600            6          TA          Y   
2     223500       1786    11250            7          Gd          Y   
3     140000       1717     9550            7          Gd          Y   
4     250000       2198    14260            8          Gd          Y   

  HeatingQC PavedDrive Neighborhood  YearBuilt  
0        Ex          Y      CollgCr       2003  
1        Ex          Y      Veenker       1976  
2        Ex          Y      CollgCr       2001  
3   

---

## 필수 1. 정규성은 충족하지만 등분산성이 위반된 경우

### 문제 1-1. 주방 품질 `Gd`와 `TA` 집단의 판매가격 비교

#### 문제 설명

주방 품질이 `Gd`(Good)인 주택과 `TA`(Typical/Average)인 주택의 판매가격을 비교하려고 합니다. 각 집단에서 30개씩 표본을 추출한 뒤 정규성과 등분산성을 확인하고 적절한 t검정 방법을 선택하세요.

#### 요구사항

1. `KitchenQual == "Gd"`인 주택의 `SalePrice`에서 30개를 추출해 `group_gd`에 저장하세요.
2. `KitchenQual == "TA"`인 주택의 `SalePrice`에서 30개를 추출해 `group_ta`에 저장하세요.
3. 두 집단의 표본 수와 평균을 확인하세요.
4. 각 집단에 Shapiro-Wilk 정규성 검정을 수행하세요.
5. 두 집단에 Levene 등분산 검정을 수행하세요.
6. 각 가정의 p-value를 0.05와 비교하여 충족 여부를 판단하세요.
7. 다음 규칙에 따라 t검정 방법을 선택하세요.
   - 두 집단 모두 정규성 충족 + 등분산성 충족: 독립표본 t검정
   - 두 집단 모두 정규성 충족 + 등분산성 위반: Welch t검정
8. 선택한 검정을 실행하고 검정통계량과 p-value를 출력하세요.
9. 두 집단의 판매가격 차이가 통계적으로 유의한지 해석하세요.

#### 해석 질문

**Q1.** Shapiro-Wilk 검정의 귀무가설은 무엇인가요?  
**Q2.** Levene 검정의 귀무가설은 무엇인가요?  
**Q3.** 가정 점검 결과 어떤 t검정 방법을 선택해야 하나요?  
**Q4.** 선택한 검정의 결과에 따르면 두 집단의 판매가격 차이는 유의한가요?

#### 제출 결과

- 집단별 표본 수와 평균
- 정규성 및 등분산성 검정 결과
- t검정 방법 선택과 선택 근거
- 최종 검정통계량과 p-value
- 결과 해석
- Q1~Q4 답변


In [2]:
# 필수 1 코드를 작성하세요.
import pandas as pd
import numpy as np
from scipy import stats

df = pd.read_csv("ames_housing.csv")

# 1, 2. 표본 추출 (random_state 고정 -> 재현 가능하게)
group_gd = df.loc[df['KitchenQual'] == 'Gd', 'SalePrice'].sample(30, random_state=42)
group_ta = df.loc[df['KitchenQual'] == 'TA', 'SalePrice'].sample(30, random_state=42)

# 3. 표본 수와 평균 확인
print(f"group_gd 표본 수: {len(group_gd)}, 평균: {group_gd.mean():.2f}")
print(f"group_ta 표본 수: {len(group_ta)}, 평균: {group_ta.mean():.2f}")

# 4. 정규성 검정 (Shapiro-Wilk)
stat_gd, p_gd = stats.shapiro(group_gd)
stat_ta, p_ta = stats.shapiro(group_ta)
print(f"\nShapiro-Wilk (group_gd): 통계량={stat_gd:.4f}, p-value={p_gd:.4f}")
print(f"Shapiro-Wilk (group_ta): 통계량={stat_ta:.4f}, p-value={p_ta:.4f}")

# 6. 정규성 가정 충족 여부 판단
normal_gd = p_gd >= 0.05
normal_ta = p_ta >= 0.05
print(f"group_gd 정규성 충족(p>=0.05): {normal_gd}")
print(f"group_ta 정규성 충족(p>=0.05): {normal_ta}")

# 5. 등분산 검정 (Levene)
stat_levene, p_levene = stats.levene(group_gd, group_ta)
print(f"\nLevene 등분산 검정: 통계량={stat_levene:.4f}, p-value={p_levene:.4f}")

# 6. 등분산성 가정 충족 여부 판단
equal_var = p_levene >= 0.05
print(f"등분산성 충족(p>=0.05): {equal_var}")

# 7, 8. 규칙에 따라 t검정 방법 선택 및 실행
if normal_gd and normal_ta:
    if equal_var:
        t_stat, p_value = stats.ttest_ind(group_gd, group_ta, equal_var=True)
        test_name = "독립표본 t검정 (등분산 가정)"
    else:
        t_stat, p_value = stats.ttest_ind(group_gd, group_ta, equal_var=False)
        test_name = "Welch t검정 (등분산 가정 X)"
else:
    # 과제 규칙에는 없는 경우지만, 정규성이 깨졌을 때 참고로 남겨둠
    print("\n[참고] 정규성 가정이 깨졌습니다. 실제로는 Mann-Whitney U 검정 같은 비모수 검정을 고려하는 게 일반적입니다.")
    t_stat, p_value = stats.ttest_ind(group_gd, group_ta, equal_var=equal_var)
    test_name = "독립표본 t검정 (참고용)"

print(f"\n선택된 검정: {test_name}")
print(f"검정통계량: {t_stat:.4f}")
print(f"p-value: {p_value:.4f}")

# 9. 해석
alpha = 0.05
if p_value < alpha:
    print(f"\np-value({p_value:.4f}) < alpha({alpha}) → 귀무가설 기각")
    print("두 집단(KitchenQual Gd vs TA)의 판매가격 평균 차이는 통계적으로 유의합니다.")
else:
    print(f"\np-value({p_value:.4f}) >= alpha({alpha}) → 귀무가설 기각 실패")
    print("두 집단의 판매가격 평균 차이가 통계적으로 유의하다고 볼 수 없습니다.")

group_gd 표본 수: 30, 평균: 190591.07
group_ta 표본 수: 30, 평균: 133586.67

Shapiro-Wilk (group_gd): 통계량=0.9348, p-value=0.0661
Shapiro-Wilk (group_ta): 통계량=0.9669, p-value=0.4571
group_gd 정규성 충족(p>=0.05): True
group_ta 정규성 충족(p>=0.05): True

Levene 등분산 검정: 통계량=6.0628, p-value=0.0168
등분산성 충족(p>=0.05): False

선택된 검정: Welch t검정 (등분산 가정 X)
검정통계량: 4.4170
p-value: 0.0001

p-value(0.0001) < alpha(0.05) → 귀무가설 기각
두 집단(KitchenQual Gd vs TA)의 판매가격 평균 차이는 통계적으로 유의합니다.


### 필수 1 답변 작성란

**Q1.** Shapiro-Wilk 검정의 귀무가설은 무엇인가요? 
-> 표본이 나온 모집단의 분포가 정규분포다.
-> p-value가 0.05 이하면 이 가설을 기각하고 정규성 위반 근거가 있다고 본다.

**Q2.** Levene 검정의 귀무가설은 무엇인가요?  
-> 두 집단의 분산이 같다
-> 두 집단의 가격이 각 집단 평균 주변에 흩어진 정도가 같은지 확인한다.

**Q3.** 가정 점검 결과 어떤 t검정 방법을 선택해야 하나요?  
-> welch t 검정을 선택, 정규성 위반 근거는 없고 levene p-value가 0.017로 등분산성 위반 근거가 있음

**Q4.** 선택한 검정의 결과에 따르면 두 집단의 판매가격 차이는 유의한가요?
-> 네. welch t검정의 p-value가 약 0.05보다 작으므로 두 집단의 모집단 평균 판매가격이 같다는 귀무가설을 기각함
-> 정규성 위반 근거가 없고 등분산성 위반 근거가 있으면 welch t검정을 선택함 

---

## 필수 2. 정규성과 등분산성이 모두 충족된 경우

### 문제 2-1. 난방 품질 `TA`와 `Fa` 집단의 판매가격 비교

#### 문제 설명

난방 품질이 `TA`(Typical/Average)인 주택과 `Fa`(Fair)인 주택의 판매가격을 비교하려고 합니다. 각 집단에서 20개씩 표본을 추출한 뒤 정규성과 등분산성을 확인하고 적절한 t검정 방법을 선택하세요.

#### 요구사항

1. `HeatingQC == "TA"`와 `HeatingQC == "Fa"`인 집단에서 `SalePrice`를 20개씩 추출하세요.
2. 두 집단의 표본 수와 평균을 확인하세요.
3. 두 집단이 독립집단인지 대응집단인지 판단하세요.
4. 각 집단에 Shapiro-Wilk 정규성 검정을 수행하세요.
5. 두 집단에 Levene 등분산 검정을 수행하세요.
6. 두 집단 모두 정규성을 충족하는지 확인하세요.
7. 등분산성 결과에 따라 독립표본 t검정 또는 Welch t검정 중 적절한 방법을 선택하세요.
8. 선택한 검정을 실행하고 검정통계량과 p-value를 출력하세요.
9. 두 집단의 판매가격 차이가 통계적으로 유의한지 해석하세요.

#### 해석 질문

**Q1.** 두 집단은 독립집단인가요, 대응집단인가요?  
**Q2.** 두 집단의 정규성 가정은 충족되나요?  
**Q3.** 두 집단의 등분산성 가정은 충족되나요?  
**Q4.** 가정 점검 결과 어떤 t검정 방법을 선택해야 하나요?  
**Q5.** 최종 검정 결과는 무엇을 의미하나요?

#### 제출 결과

- 집단별 표본 수와 평균
- 집단 관계 판단
- 정규성 및 등분산성 검정 결과
- t검정 방법과 선택 근거
- 검정통계량과 p-value
- 결과 해석
- Q1~Q5 답변


In [3]:
# 필수 2 코드를 작성하세요.
import pandas as pd
import numpy as np
from scipy import stats

df = pd.read_csv("ames_housing.csv")

# 1. 표본 추출 (20개씩)
group_ta = df.loc[df['HeatingQC'] == 'TA', 'SalePrice'].sample(20, random_state=42)
group_fa = df.loc[df['HeatingQC'] == 'Fa', 'SalePrice'].sample(20, random_state=42)

# 2. 표본 수와 평균 확인
print(f"group_ta 표본 수: {len(group_ta)}, 평균: {group_ta.mean():.2f}")
print(f"group_fa 표본 수: {len(group_fa)}, 평균: {group_fa.mean():.2f}")

# 3. 독립/대응 판단
print("\n두 집단은 서로 다른 주택(서로 다른 행)에서 추출된, 서로 무관한 표본이므로 독립집단입니다.")

# 4. 정규성 검정 (Shapiro-Wilk)
stat_ta, p_ta = stats.shapiro(group_ta)
stat_fa, p_fa = stats.shapiro(group_fa)
print(f"\nShapiro-Wilk (group_ta): 통계량={stat_ta:.4f}, p-value={p_ta:.4f}")
print(f"Shapiro-Wilk (group_fa): 통계량={stat_fa:.4f}, p-value={p_fa:.4f}")

# 6. 정규성 충족 여부
normal_ta = p_ta >= 0.05
normal_fa = p_fa >= 0.05
print(f"group_ta 정규성 충족(p>=0.05): {normal_ta}")
print(f"group_fa 정규성 충족(p>=0.05): {normal_fa}")

# 5. 등분산 검정 (Levene)
stat_levene, p_levene = stats.levene(group_ta, group_fa)
print(f"\nLevene 등분산 검정: 통계량={stat_levene:.4f}, p-value={p_levene:.4f}")
equal_var = p_levene >= 0.05
print(f"등분산성 충족(p>=0.05): {equal_var}")

# 7, 8. 검정 방법 선택 및 실행
if normal_ta and normal_fa:
    if equal_var:
        t_stat, p_value = stats.ttest_ind(group_ta, group_fa, equal_var=True)
        test_name = "독립표본 t검정 (등분산 가정)"
    else:
        t_stat, p_value = stats.ttest_ind(group_ta, group_fa, equal_var=False)
        test_name = "Welch t검정 (등분산 가정 X)"
else:
    print("\n[참고] 정규성 가정이 깨졌습니다. 실제로는 비모수검정(Mann-Whitney U)을 고려해야 합니다.")
    t_stat, p_value = stats.ttest_ind(group_ta, group_fa, equal_var=equal_var)
    test_name = "독립표본 t검정 (참고용)"

print(f"\n선택된 검정: {test_name}")
print(f"검정통계량: {t_stat:.4f}")
print(f"p-value: {p_value:.4f}")

# 9. 해석
alpha = 0.05
if p_value < alpha:
    print(f"\np-value({p_value:.4f}) < alpha({alpha}) → 귀무가설 기각")
    print("두 집단(HeatingQC TA vs Fa)의 판매가격 평균 차이는 통계적으로 유의합니다.")
else:
    print(f"\np-value({p_value:.4f}) >= alpha({alpha}) → 귀무가설 기각 실패")
    print("두 집단의 판매가격 평균 차이가 통계적으로 유의하다고 볼 수 없습니다.")


group_ta 표본 수: 20, 평균: 130845.00
group_fa 표본 수: 20, 평균: 122855.00

두 집단은 서로 다른 주택(서로 다른 행)에서 추출된, 서로 무관한 표본이므로 독립집단입니다.

Shapiro-Wilk (group_ta): 통계량=0.9736, p-value=0.8290
Shapiro-Wilk (group_fa): 통계량=0.9319, p-value=0.1681
group_ta 정규성 충족(p>=0.05): True
group_fa 정규성 충족(p>=0.05): True

Levene 등분산 검정: 통계량=1.3104, p-value=0.2595
등분산성 충족(p>=0.05): True

선택된 검정: 독립표본 t검정 (등분산 가정)
검정통계량: 0.5584
p-value: 0.5799

p-value(0.5799) >= alpha(0.05) → 귀무가설 기각 실패
두 집단의 판매가격 평균 차이가 통계적으로 유의하다고 볼 수 없습니다.


### 필수 2 답변 작성란

**Q1.** 두 집단은 독립집단인가요, 대응집단인가요? 
-> 필수 2번에서는 독립집단, 서로 다른 주택을 비교하여 같은 주택의 리모델링 전후처럼 관측값이 짝을 이루는 대응집단이 아님

**Q2.** 두 집단의 정규성 가정은 충족되나요?  
-> 두 집단 모두 shapiro p-value가 0.05보다 크므로 정규성 위반 근거가 부족함

**Q3.** 두 집단의 등분산성 가정은 충족되나요?  
-> Levene p-value가 약 0.259이므로 0.05보다 크므로 등분산성 위반 근거가 부족함

**Q4.** 가정 점검 결과 어떤 t검정 방법을 선택해야 하나요?  
-> 실습의 선택 기준에 따라 등분산을 가정한 독립표본 t검정을 선택

**Q5.** 최종 검정 결과는 무엇을 의미하나요?
-> p-value가 약 0.58이므로 0.05보다 크다
-> 평균의 차이가 통계적으로 유의하지 않음

-> 필수 2번에서 정규성과 등분산성의 위반 근거가 없으면 등분산을 가정한 독립표본 t검정을 선택

---

## 과제. 난방 품질에 따른 t검정 방법 선택

### 문제 3-1. 난방 품질 `Ex`와 `TA` 집단의 판매가격 비교

#### 문제 설명

난방 품질이 `Ex`(Excellent)인 주택과 `TA`(Typical/Average)인 주택의 판매가격을 비교하려고 합니다. 각 집단에서 20개씩 표본을 추출한 뒤 필수 문제에서 학습한 **독립표본 t검정과 Welch t검정 선택 과정**을 독립적으로 적용하세요.

#### 요구사항

1. `HeatingQC == "Ex"`인 집단과 `HeatingQC == "TA"`인 집단에서 `SalePrice`를 20개씩 추출하세요.
2. 두 집단의 표본 수와 평균을 출력하세요.
3. 두 집단이 독립집단인지 대응집단인지 판단하세요.
4. 각 집단의 정규성과 두 집단의 등분산성을 검정하세요.
5. 두 집단 모두 정규성을 충족하는지 확인하세요.
6. 등분산성 결과에 따라 독립표본 t검정 또는 Welch t검정 중 적절한 방법을 선택하고 선택 이유를 작성하세요.
7. 선택한 검정을 실행하여 검정통계량과 p-value를 출력하세요.
8. 난방 품질에 따라 판매가격에 유의한 차이가 있는지 결론을 작성하세요.

#### 해석 질문

**Q1.** 두 집단의 정규성 가정은 충족되나요?  
**Q2.** 두 집단의 등분산성 가정은 충족되나요?  
**Q3.** 최종적으로 어떤 t검정 방법을 선택해야 하나요?  
**Q4.** 검정 결과 난방 품질에 따른 판매가격 차이는 통계적으로 유의한가요?

#### 제출 결과

- 표본 구성과 집단 관계 판단
- 정규성 및 등분산성 검정 결과
- 최종 t검정 선택과 근거
- 검정통계량과 p-value
- 결과 해석
- Q1~Q4 답변


In [5]:
# 과제 코드를 작성하세요.

import pandas as pd
import numpy as np
from scipy import stats

df = pd.read_csv("ames_housing.csv")

# 1. 표본 추출 (20개씩)
group_ex = df.loc[df['HeatingQC'] == 'Ex', 'SalePrice'].sample(20, random_state=42)
group_ta = df.loc[df['HeatingQC'] == 'TA', 'SalePrice'].sample(20, random_state=42)

# 2. 표본 수와 평균 확인
print(f"group_ex 표본 수: {len(group_ex)}, 평균: {group_ex.mean():.2f}")
print(f"group_ta 표본 수: {len(group_ta)}, 평균: {group_ta.mean():.2f}")

# 3. 독립/대응 판단
print("\n서로 다른 주택(서로 다른 행)에서 추출된 무관한 표본이므로 독립집단입니다.")

# 4. 정규성 검정 (Shapiro-Wilk)
stat_ex, p_ex = stats.shapiro(group_ex)
stat_ta, p_ta = stats.shapiro(group_ta)
print(f"\nShapiro-Wilk (group_ex): 통계량={stat_ex:.4f}, p-value={p_ex:.4f}")
print(f"Shapiro-Wilk (group_ta): 통계량={stat_ta:.4f}, p-value={p_ta:.4f}")

# 4. 등분산 검정 (Levene)
stat_levene, p_levene = stats.levene(group_ex, group_ta)
print(f"\nLevene 등분산 검정: 통계량={stat_levene:.4f}, p-value={p_levene:.4f}")

# 5. 정규성/등분산성 충족 여부
normal_ex = p_ex >= 0.05
normal_ta = p_ta >= 0.05
print(f"\ngroup_ex 정규성 충족(p>=0.05): {normal_ex}")
print(f"group_ta 정규성 충족(p>=0.05): {normal_ta}")

equal_var = p_levene >= 0.05
print(f"등분산성 충족(p>=0.05): {equal_var}")

# 6, 7. 검정 방법 선택 + 이유 + 실행
if normal_ex and normal_ta:
    if equal_var:
        t_stat, p_value = stats.ttest_ind(group_ex, group_ta, equal_var=True)
        test_name = "독립표본 t검정 (Student's t-test)"
        reason = (f"두 집단 모두 정규성을 충족했고(p_ex={p_ex:.4f}, p_ta={p_ta:.4f} 모두 >=0.05), "
                  f"Levene 검정에서도 등분산성이 충족되어(p={p_levene:.4f}>=0.05) "
                  f"분산을 하나로 합쳐 계산하는 독립표본 t검정을 선택했습니다.")
    else:
        t_stat, p_value = stats.ttest_ind(group_ex, group_ta, equal_var=False)
        test_name = "Welch t검정"
        reason = (f"두 집단 모두 정규성은 충족했지만(p_ex={p_ex:.4f}, p_ta={p_ta:.4f} 모두 >=0.05), "
                  f"Levene 검정에서 등분산성이 위반되어(p={p_levene:.4f}<0.05) "
                  f"두 집단의 분산을 각각 반영하는 Welch t검정을 선택했습니다.")
else:
    print("\n[참고] 정규성 가정이 깨졌습니다. 실제로는 비모수검정(Mann-Whitney U)을 고려해야 합니다.")
    t_stat, p_value = stats.ttest_ind(group_ex, group_ta, equal_var=equal_var)
    test_name = "독립표본 t검정 (참고용)"
    reason = "정규성 가정이 위반되어 참고용으로만 계산했습니다."

print(f"\n선택된 검정: {test_name}")
print(f"선택 이유: {reason}")
print(f"검정통계량: {t_stat:.4f}")
print(f"p-value: {p_value:.4e}")

# 8. 결론
alpha = 0.05
if p_value < alpha:
    print(f"\np-value({p_value:.4f}) < alpha({alpha}) → 귀무가설 기각")
    print("난방 품질(Ex vs TA)에 따라 판매가격에 통계적으로 유의한 차이가 있습니다.")
else:
    print(f"\np-value({p_value:.4f}) >= alpha({alpha}) → 귀무가설 기각 실패")
    print("난방 품질에 따른 판매가격 차이가 통계적으로 유의하다고 볼 수 없습니다.")

group_ex 표본 수: 20, 평균: 259451.20
group_ta 표본 수: 20, 평균: 130845.00

서로 다른 주택(서로 다른 행)에서 추출된 무관한 표본이므로 독립집단입니다.

Shapiro-Wilk (group_ex): 통계량=0.9447, p-value=0.2939
Shapiro-Wilk (group_ta): 통계량=0.9736, p-value=0.8290

Levene 등분산 검정: 통계량=6.9271, p-value=0.0122

group_ex 정규성 충족(p>=0.05): True
group_ta 정규성 충족(p>=0.05): True
등분산성 충족(p>=0.05): False

선택된 검정: Welch t검정
선택 이유: 두 집단 모두 정규성은 충족했지만(p_ex=0.2939, p_ta=0.8290 모두 >=0.05), Levene 검정에서 등분산성이 위반되어(p=0.0122<0.05) 두 집단의 분산을 각각 반영하는 Welch t검정을 선택했습니다.
검정통계량: 4.9073
p-value: 4.9458e-05

p-value(0.0000) < alpha(0.05) → 귀무가설 기각
난방 품질(Ex vs TA)에 따라 판매가격에 통계적으로 유의한 차이가 있습니다.


### 과제 답변 작성란

**Q1.** 두 집단의 정규성 가정은 충족되나요?  
-> 충족된다고 볼 수 있음
-> Shapiro-Wilk 검정시 , group_ex는 p=0.2939, group_ta는 p=0.8290
-> 두 집단 모두 shapiro p-value가 0.05보다 커서 귀무가설을 기각하지 못함
-> 정규성 가정을 위반했다고 볼 근거가 없기때문에 충족된다고 볼 수 있음

**Q2.** 두 집단의 등분산성 가정은 충족되나요?  
-> 충족하지 않음
-> Levene 검정시, p=0.0122로 0.05보다 작아 귀무가설을 기각함
-> 두 집단의 등분산성에 통계적으로 유의한 차이가 있다는 것을 근거로 볼 수 있음

**Q3.** 최종적으로 어떤 t검정 방법을 선택해야 하나요?  
-> Welch t검정
-> 두 집단 모두 정규성은 충족했지만 등분산성은 귀무가설을 기각했기 때문

**Q4.** 검정 결과 난방 품질에 따른 판매가격 차이는 통계적으로 유의한가요?
-> 유의하다고 볼 수 있음
-> p=4.9458e-05로 0.05보다 작아 귀무가설을 기각함
-> 두 집단의 평균에 통계적으로 유의한 차이가 있다는 것을 근거로 볼 수 있음

---

## 실습 마무리

1. 두 집단을 비교하기 전에 어떤 데이터 특성을 먼저 확인해야 하나요?
-> 비교할 값은 판매가격처럼 수치형 변수인지, 두 집단은 독립, 대응인지 확인해야함

2. 정규성 검정과 등분산 검정에서 `p > 0.05`는 무엇을 의미하나요?
-> shapiro 검정은 정규성, levene검정은 등분산성, t검정은 두 모집단의 평균차이를 확인

3. 두 집단 모두 정규성을 충족하고 등분산성도 충족하면 어떤 검정을 사용할 수 있나요?
-> 정규성, 등분산성 검정에서 p > 0.05이면 해당 가정의 위반 근거가 부족하다 라는 뜻
-> 가정이 참이라고 증명된 것음 아님
-> 독립표본 t검정을 선택

4. 두 집단 모두 정규성을 충족하지만 등분산성이 위반되면 어떤 검정을 사용할 수 있나요?
-> welch t검정을 선택

5. 독립표본 t검정과 Welch t검정을 선택할 때 정규성과 등분산성을 함께 확인해야 하는 이유는 무엇인가요?
-> 정규성 평균에 퍼짐 정도 일반적으로 예상이 가능한 범주
-> 등분산성 두 집단 사이가 얼마나 유사한지